# AlphaNetV4: Alpha Mining Model

**论文参考**: Wenjun Wu (2024), *AlphaNetV4: Alpha Mining Model*, IC mean 8.08%, IR 1.52

本notebook实现Bi-LSTM + Transformer Encoder混合架构，利用Bi-LSTM提取局部时序特征后送入Transformer捕捉全局依赖，预测贵州茅台(600519)次日涨跌方向。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### AlphaNetV4 架构

**Stage 1: Bi-LSTM 局部特征提取**

双向LSTM同时从过去和未来方向读取序列:
$$\overrightarrow{h_t} = \text{LSTM}_{\text{forward}}(x_t, \overrightarrow{h_{t-1}})$$
$$\overleftarrow{h_t} = \text{LSTM}_{\text{backward}}(x_t, \overleftarrow{h_{t+1}})$$
$$h_t = [\overrightarrow{h_t}; \overleftarrow{h_t}]$$

**Stage 2: Transformer Encoder 全局依赖建模**

将Bi-LSTM输出作为Transformer输入:
$$Z = \text{TransformerEncoder}(\text{PE}(H_{\text{BiLSTM}}))$$

**输出层**:
$$\hat{y} = \sigma(W_{out} \cdot z_{T} + b_{out})$$

### 评价指标

**Information Coefficient (IC)**:
$$IC = \text{Corr}(\hat{y}, y_{\text{actual}})$$

**Information Ratio (IR)**:
$$IR = \frac{\text{mean}(IC)}{\text{std}(IC)}$$

### 损失函数
$$\mathcal{L} = \text{BCE}(\hat{y}, y) + \lambda \|\theta\|_2^2$$

In [ ]:
# ===== Cell 3: 架构流图动画 (Bi-LSTM -> Transformer) =====
import sys
sys.path.insert(0, '../shared')
from animation_helpers import create_architecture_diagram

layers = [
    ("Input\n(30x6)", 6),
    ("Bi-LSTM\n(128)", 128),
    ("Transformer\nEncoder\n(2 layers)", 128),
    ("FC\n(64)", 64),
    ("Output\n(1)", 1),
]

fig = create_architecture_diagram(layers, title="AlphaNetV4: Bi-LSTM → Transformer 混合架构")
fig.show()

In [ ]:
# ===== Cell 4: Imports & Setup =====
import sys
sys.path.insert(0, '../shared')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import math
from torch.utils.data import DataLoader, TensorDataset
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from data_fetcher import get_stock_daily
from factors import rsi, macd, bollinger_bands, volatility
from backtest_engine import Backtester
from visualization import plot_equity_curve, plot_drawdown, plot_metrics_table
from constants import STOCK_MOUTAI, INITIAL_CAPITAL

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ===== Cell 5: 数据获取 =====
df = get_stock_daily(STOCK_MOUTAI, start_date="20200101", end_date="20241231")
print(f"数据形状: {df.shape}, 日期范围: {df.index[0]} ~ {df.index[-1]}")
df.head()

In [ ]:
# ===== Cell 6: 特征工程 & 滑动窗口 =====
df['close_pct'] = df['close'].pct_change()
df['volume_pct'] = df['volume'].pct_change()
df['rsi'] = rsi(df['close'], window=14)
macd_df = macd(df['close'])
df['macd_hist'] = macd_df['histogram']
bb_df = bollinger_bands(df['close'])
df['bb_pctb'] = bb_df['bb_pctb']
vol_df = volatility(df['close'], windows=[20])
df['volatility'] = vol_df['vol_20']
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)

feature_cols = ['close_pct', 'volume_pct', 'rsi', 'macd_hist', 'bb_pctb', 'volatility']
df = df.dropna(subset=feature_cols + ['target'])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

train_mask = df.index < '2024-01-01'
test_mask = df.index >= '2024-01-01'
train_data = df[train_mask].copy()
test_data = df[test_mask].copy()

scaler.fit(train_data[feature_cols])
train_data[feature_cols] = scaler.transform(train_data[feature_cols])
test_data[feature_cols] = scaler.transform(test_data[feature_cols])

WINDOW = 30

def create_sequences(data, feature_cols, target_col, window):
    X, y, dates = [], [], []
    features = data[feature_cols].values
    targets = data[target_col].values
    idx = data.index
    for i in range(window, len(data)):
        X.append(features[i-window:i])
        y.append(targets[i])
        dates.append(idx[i])
    return np.array(X), np.array(y), dates

X_train, y_train, dates_train = create_sequences(train_data, feature_cols, 'target', WINDOW)
X_test, y_test, dates_test = create_sequences(test_data, feature_cols, 'target', WINDOW)

print(f"训练集: X={X_train.shape}, y={y_train.shape}")
print(f"测试集: X={X_test.shape}, y={y_test.shape}")

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)),
                          batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test)),
                         batch_size=32, shuffle=False)

In [ ]:
# ===== Cell 7: AlphaNetV4 模型定义 & 训练 =====

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model > 1:
            pe[:, 1::2] = torch.cos(position * div_term[:d_model//2])
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class AlphaNetV4(nn.Module):
    """Bi-LSTM -> Transformer Encoder 混合架构"""
    def __init__(self, input_size=6, lstm_hidden=64, d_model=128,
                 nhead=4, num_transformer_layers=2, dropout=0.1):
        super().__init__()
        # Stage 1: Bi-LSTM
        self.bilstm = nn.LSTM(input_size, lstm_hidden, num_layers=1,
                              batch_first=True, bidirectional=True)
        # lstm_hidden * 2 因为双向
        bilstm_out = lstm_hidden * 2
        
        # 投影到 d_model
        self.proj = nn.Linear(bilstm_out, d_model)
        
        # Stage 2: Transformer Encoder
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)
        
        # Output
        self.fc = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        # x: (B, T, input_size)
        bilstm_out, _ = self.bilstm(x)    # (B, T, lstm_hidden*2)
        proj = self.proj(bilstm_out)        # (B, T, d_model)
        proj = self.pos_encoder(proj)
        trans_out = self.transformer(proj)   # (B, T, d_model)
        out = trans_out[:, -1, :]            # (B, d_model)
        return torch.sigmoid(self.fc(out)).squeeze(-1)


model = AlphaNetV4(input_size=6, lstm_hidden=64, d_model=128,
                   nhead=4, num_transformer_layers=2, dropout=0.1).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60)
criterion = nn.BCELoss()

history = []
for epoch in range(60):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    history.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/60, Loss: {avg_loss:.4f}")

# 预测
model.eval()
preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        pred = model(X_batch.to(device))
        preds.extend(pred.cpu().numpy())
preds = np.array(preds)

acc = ((preds > 0.5).astype(int) == y_test).mean()
print(f"\n测试集准确率: {acc:.4f}")

In [ ]:
# ===== Cell 8: 信号生成 & 回测 =====
test_prices = df.loc[dates_test, 'close']
signals = pd.Series((preds > 0.5).astype(int), index=dates_test)

bt = Backtester(initial_capital=INITIAL_CAPITAL)
result = bt.run(test_prices, signals)

print("AlphaNetV4 回测结果:")
for k, v in result['metrics'].items():
    print(f"  {k}: {v}")

In [ ]:
# ===== Cell 9: 可视化 =====

# 1. 训练损失
fig = go.Figure()
fig.add_trace(go.Scatter(y=history, mode='lines', name='训练损失',
                         line=dict(color='#9C27B0', width=2)))
fig.update_layout(title='AlphaNetV4 训练损失 (CosineAnnealing LR)',
                  xaxis_title='Epoch', yaxis_title='BCE Loss', height=350, width=700)
fig.show()

# 2. 预测概率 vs 实际
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['预测概率', '收盘价与信号'])
fig.add_trace(go.Scatter(x=dates_test, y=preds, mode='lines',
                         name='预测概率', line=dict(color='#9C27B0')), row=1, col=1)
fig.add_hline(y=0.5, line_dash='dash', line_color='red', row=1, col=1)

buy_mask = signals.diff().fillna(0) > 0
sell_mask = signals.diff().fillna(0) < 0
fig.add_trace(go.Scatter(x=test_prices.index, y=test_prices.values,
                         mode='lines', name='收盘价', line=dict(color='#333')), row=2, col=1)
fig.add_trace(go.Scatter(x=test_prices.index[buy_mask], y=test_prices[buy_mask].values,
                         mode='markers', name='买入',
                         marker=dict(color='green', size=8, symbol='triangle-up')), row=2, col=1)
fig.add_trace(go.Scatter(x=test_prices.index[sell_mask], y=test_prices[sell_mask].values,
                         mode='markers', name='卖出',
                         marker=dict(color='red', size=8, symbol='triangle-down')), row=2, col=1)
fig.update_layout(height=600, width=1000, title='AlphaNetV4 预测与交易信号')
fig.show()

# 3. 权益曲线
plot_equity_curve(result['equity_curve'], title='AlphaNetV4 策略收益曲线')

# 4. 回撤
plot_drawdown(result['equity_curve'], title='AlphaNetV4 策略回撤')

# 5. 绩效表
plot_metrics_table(result['metrics'], title='AlphaNetV4 绩效指标')

## 结果讨论

### AlphaNetV4 架构优势
1. **Bi-LSTM**: 双向读取序列，同时捕捉过去趋势和近期反转模式
2. **Transformer**: 在Bi-LSTM提取的局部特征上建模全局依赖
3. **两阶段设计**: 避免纯Transformer在短序列上的位置编码不足问题

### IC/IR 分析
- 论文报告IC mean 8.08%, IR 1.52，本实验在单股票上的IC表现受限于样本量
- 横截面因子选股场景下，混合架构的IC表现更优

### 改进方向
- 扩展到多股票横截面预测
- 加入行业/市值因子进行中性化
- 使用IC加权集成多个模型预测

### 参考
- Wu, W. (2024). AlphaNetV4: Alpha Mining Model.